# Qəbz oxuyucusu: vLLM multi-LoRA serve (ixtiyari, öyrətmədən SONRA)

Əvvəl `ml/qlora_receipts.ipynb` işləməlidir: adapter `MyDrive/masterai-receipts/adapter`-dədir.
Bunu **yeni sessiyada** işlət (vLLM öz torch versiyasını qoyur). `--enable-lora --lora-modules
receipts=...`: sorğuda `"model": "receipts"` adapter, baza adı isə baza model deməkdir.

In [ ]:
# 1) vLLM + repo + Drive
!pip -q install "vllm==0.6.6.post1" httpx
!rm -rf masterai && git clone -q --depth 1 https://github.com/Alihuseyn-ibrahimov/masterai.git
import json, os, secrets, subprocess, sys, time, urllib.request
sys.path.insert(0, "/content/masterai")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/masterai-receipts"
ADAPTER = f"{DRIVE}/adapter"
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "Əvvəl ml/qlora_receipts.ipynb işlət"

In [ ]:
# 2) Server
API_KEY = secrets.token_urlsafe(24)
log = open("vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--max-model-len", "2048",
    "--enable-lora", "--lora-modules", f"receipts={ADAPTER}", "--max-lora-rank", "16",
    "--gpu-memory-utilization", "0.85", "--api-key", API_KEY, "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("vLLM hazırdır"); break
    except Exception:
        time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

## Məcburi JSON (`guided_json`)
Dekodlaşdırma produksiyadakı `Receipt` sxeminə bağlanır: JSON oxunma 100% olmalıdır, sual məzmundur.
Eyni `evals.kassa` dəsti, nəticə `report_vllm.md`-yə yazılır.

In [ ]:
import httpx
from IPython.display import Markdown, display
from ml.receipts import compare
from pipeline.receipt_reader import Receipt

SCHEMA = Receipt.model_json_schema()
parts = compare.eval_parts()
prompts = compare.prompts(parts)

def ask(row, model="receipts", guided=False):
    body = {"model": model, "max_tokens": 160, "temperature": 0, "messages": row["messages"]}
    if guided:
        body["guided_json"] = SCHEMA
    r = httpx.post("http://localhost:8000/v1/chat/completions", json=body, timeout=120,
                   headers={"Authorization": f"Bearer {API_KEY}"})
    return r.json()["choices"][0]["message"]["content"]

runs = {"vllm_qlora": {}, "vllm_qlora_guided": {"guided": True},
        "vllm_base_guided": {"guided": True, "model": "Qwen/Qwen2.5-1.5B-Instruct"}}
outputs = {name: {r["text"]: ask(r, **kw) for r in prompts} for name, kw in runs.items()}
report = compare.compare(outputs, parts)
compare.save(report, f"{DRIVE}/report_vllm.md")
display(Markdown(compare.format_report(report)))

## Sürət
Qəbz qısadır: əsas göstərici gecikmədir (TTFT, p95). `ml/bench_llm.py` ilə 1/4/8 paralel.

In [ ]:
!cd /content/masterai && python -m ml.receipts.sft --per-family 100 --seed 7 --out /content/receipts
!cd /content/masterai && python -m ml.bench_llm --url http://localhost:8000 --model receipts --api-key {API_KEY} \
    --val /content/receipts/test_unseen.jsonl --max-tokens 160 --concurrency 1,4,8 --requests 32 --out {DRIVE}/bench.json

## MasterAI serverinə qoş
Cloudflare tuneli. Colab bağlananda ünvan ölür: server qaydalı oxuyucuya düşür, Kassa işləyir.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if match:
        url = match.group(0); break
print("Serverdə /opt/masterai/.env-ə yaz və `systemctl restart masterai`:\n")
print(f"VLLM_BASE_URL={url}\nVLLM_API_KEY={API_KEY}\nVLLM_RECEIPT_MODEL=receipts")